In [1]:
import requests 
import zipfile
import io

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [2]:
import os

extracted_files = os.listdir("/home/nnorian/htb-ai/skills_assessment_data")
print("extracted files:", extracted_files)

extracted files: ['test.json', '.ipynb_checkpoints', 'train.json']


In [3]:
import json

with open("./skills_assessment_data/train.json") as f:
    data = json.load(f)

print(type(data))
if isinstance(data, list):
    print(data[:2])  # show first 2 entries
elif isinstance(data, dict):
    print(list(data.keys()))
    first_key = list(data.keys())[0]
    print(type(data[first_key]), data[first_key][:2] if isinstance(data[first_key], list) else data[first_key])

<class 'list'>
[{'text': 'Bromwell High is a cartoon comedy. It ran at the same time as some other programs about school life, such as "Teachers". My 35 years in the teaching profession lead me to believe that Bromwell High\'s satire is much closer to reality than is "Teachers". The scramble to survive financially, the insightful students who can see right through their pathetic teachers\' pomp, the pettiness of the whole situation, all remind me of the schools I knew and their students. When I saw the episode in which a student repeatedly tried to burn down the school, I immediately recalled ......... at .......... High. A classic line: INSPECTOR: I\'m here to sack one of your teachers. STUDENT: Welcome to Bromwell High. I expect that many adults of my age think that Bromwell High is far fetched. What a pity that it isn\'t!', 'label': 1}, {'text': 'Homelessness (or Houselessness as George Carlin stated) has been an issue for years but never a plan to help those on the street that were

In [4]:
import pandas as pd

train_df = pd.read_json("./skills_assessment_data/train.json")
test_df = pd.read_json("./skills_assessment_data/test.json")

print("done")
print(train_df.shape)
print(train_df.head())

done
(25000, 2)
                                                text  label
0  Bromwell High is a cartoon comedy. It ran at t...      1
1  Homelessness (or Houselessness as George Carli...      1
2  Brilliant over-acting by Lesley Ann Warren. Be...      1
3  This is easily the most underrated film inn th...      1
4  This is not the typical Mel Brooks film. It wa...      1


In [5]:
print("HEAD")
print(train_df.head())
print("DESCRIBE")
print(train_df.describe())
print("INFO")
print(train_df.info())

HEAD
                                                text  label
0  Bromwell High is a cartoon comedy. It ran at t...      1
1  Homelessness (or Houselessness as George Carli...      1
2  Brilliant over-acting by Lesley Ann Warren. Be...      1
3  This is easily the most underrated film inn th...      1
4  This is not the typical Mel Brooks film. It wa...      1
DESCRIBE
             label
count  25000.00000
mean       0.50000
std        0.50001
min        0.00000
25%        0.00000
50%        0.50000
75%        1.00000
max        1.00000
INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    25000 non-null  object
 1   label   25000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 390.8+ KB
None


In [6]:
# check for duplicates

print("duplicate entries:", train_df.duplicated().sum())
print("duplicate entries:", test_df.duplicated().sum())

df = train_df.drop_duplicates()
df = test_df.drop_duplicates()


duplicate entries: 96
duplicate entries: 199


In [7]:
# check for overlap

overlap = set(train_df["text"]) & set(test_df["text"])
test_df = test_df[~test_df["text"].isin(overlap)].reset_index(drop=True)

In [8]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

print("BEFORE ANY PROCESSING")
print(train_df.head(5))

BEFORE ANY PROCESSING
                                                text  label
0  Bromwell High is a cartoon comedy. It ran at t...      1
1  Homelessness (or Houselessness as George Carli...      1
2  Brilliant over-acting by Lesley Ann Warren. Be...      1
3  This is easily the most underrated film inn th...      1
4  This is not the typical Mel Brooks film. It wa...      1


[nltk_data] Downloading package punkt to /home/nnorian/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/nnorian/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/nnorian/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [9]:
# making both lowercase 

train_df["text"] = train_df["text"].str.lower()
print("AFTER LOWERCASTING")
print(train_df["text"].head(5))

test_df["text"] = test_df["text"].str.lower()
print("AFTER LOWERCASTING")
print(test_df["text"].head(5))

AFTER LOWERCASTING
0    bromwell high is a cartoon comedy. it ran at t...
1    homelessness (or houselessness as george carli...
2    brilliant over-acting by lesley ann warren. be...
3    this is easily the most underrated film inn th...
4    this is not the typical mel brooks film. it wa...
Name: text, dtype: object
AFTER LOWERCASTING
0    i went and saw this movie last night after bei...
1    actor turned director bill paxton follows up h...
2    as a recreational golfer with some knowledge o...
3    i saw this film in a sneak preview, and it is ...
4    bill paxton has taken the true story of the 19...
Name: text, dtype: object


In [10]:
# removing punctuation and stuff

import re 

train_df["text"] = train_df["text"].apply(lambda x: re.sub(r"[^a-z\s$!]", "", x))
print("\n=== AFTER REMOVING PUNCTUATION & NUMBERS (except $ and !) ===")
print(train_df["text"].head(5))

test_df["text"] = test_df["text"].apply(lambda x: re.sub(r"[^a-z\s$!]", "", x))
print("\n=== AFTER REMOVING PUNCTUATION & NUMBERS (except $ and !) ===")
print(test_df["text"].head(5))


=== AFTER REMOVING PUNCTUATION & NUMBERS (except $ and !) ===
0    bromwell high is a cartoon comedy it ran at th...
1    homelessness or houselessness as george carlin...
2    brilliant overacting by lesley ann warren best...
3    this is easily the most underrated film inn th...
4    this is not the typical mel brooks film it was...
Name: text, dtype: object

=== AFTER REMOVING PUNCTUATION & NUMBERS (except $ and !) ===
0    i went and saw this movie last night after bei...
1    actor turned director bill paxton follows up h...
2    as a recreational golfer with some knowledge o...
3    i saw this film in a sneak preview and it is d...
4    bill paxton has taken the true story of the  u...
Name: text, dtype: object


In [11]:
# tokenising the stuff

from nltk.tokenize import word_tokenize

train_df["text"] = train_df["text"].apply(word_tokenize)
print("\n AFTER TOKENIZATION")
print(train_df["text"].head(5))

test_df["text"] = test_df["text"].apply(word_tokenize)
print("\n AFTER TOKENIZATION")
print(test_df["text"].head(5))


 AFTER TOKENIZATION
0    [bromwell, high, is, a, cartoon, comedy, it, r...
1    [homelessness, or, houselessness, as, george, ...
2    [brilliant, overacting, by, lesley, ann, warre...
3    [this, is, easily, the, most, underrated, film...
4    [this, is, not, the, typical, mel, brooks, fil...
Name: text, dtype: object

 AFTER TOKENIZATION
0    [i, went, and, saw, this, movie, last, night, ...
1    [actor, turned, director, bill, paxton, follow...
2    [as, a, recreational, golfer, with, some, know...
3    [i, saw, this, film, in, a, sneak, preview, an...
4    [bill, paxton, has, taken, the, true, story, o...
Name: text, dtype: object


In [13]:
# back to string 

train_df["text"] = train_df["text"].apply(lambda x: " ".join(x))
print("\n=== AFTER JOINING TOKENS BACK INTO STRINGS ===")
print(train_df["text"].head(5))

test_df["text"] = test_df["text"].apply(lambda x: " ".join(x))
print("\n=== AFTER JOINING TOKENS BACK INTO STRINGS ===")
print(test_df["text"].head(5))


=== AFTER JOINING TOKENS BACK INTO STRINGS ===
0    bromwell high is a cartoon comedy it ran at th...
1    homelessness or houselessness as george carlin...
2    brilliant overacting by lesley ann warren best...
3    this is easily the most underrated film inn th...
4    this is not the typical mel brooks film it was...
Name: text, dtype: object

=== AFTER JOINING TOKENS BACK INTO STRINGS ===
0    i went and saw this movie last night after bei...
1    actor turned director bill paxton follows up h...
2    as a recreational golfer with some knowledge o...
3    i saw this film in a sneak preview and it is d...
4    bill paxton has taken the true story of the us...
Name: text, dtype: object


In [16]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import VotingClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

vectorizer = TfidfVectorizer(ngram_range=(1,3), max_df=0.9, min_df=2, sublinear_tf=True)

voting_pipeline = Pipeline([
    ("vectorizer", vectorizer),
    ("classifier", VotingClassifier(estimators=[
        ("lr", LogisticRegression(max_iter=1000, C=10)),
        ("svm", LinearSVC(max_iter=5000, C=1)),
        ("nb", MultinomialNB(alpha=0.5))
    ], voting="hard"))
])

voting_pipeline.fit(train_df["text"], train_df["label"])
test_accuracy = voting_pipeline.score(test_df["text"], test_df["label"])
print("Test accuracy:", test_accuracy)
best_model = voting_pipeline

Test accuracy: 0.9067009687663303


In [18]:
import joblib

best_model = voting_pipeline
joblib.dump(best_model, "skills_assessment.joblib")
print("Model saved successfully")

Model saved successfully
